# OPTIMA — Kaggle: Heavy Hugging Face LLM Enrichment Experiment

Compares Hugging Face causal LMs as the **enrichment** stage of the Optima pipeline:

```
GitHub base.json (frozen snapshot)
        |
        v
  LLM enrichment (gated: load -> trivial -> 1 function -> 3 functions -> full 96)
        |
        v
  embeddings (optima.rag, unchanged) -> FAISS indexes
        |
        v
  retrieval (unchanged) -> Recall@K / MRR evaluation
```

**How to use this notebook**

1. Edit the `CONFIG` cell: `BASE_JSON_URL`, `ACTIVE_MODEL`, and (optionally) `MODEL_QUEUE`.
2. *Run All*. Each model must pass GATES 1-4 before the full 96-function run starts.
3. For a run that may take hours, use **Save & Run All (Commit)** so the checkpoint
   survives a lost session; resume with `RESUME_INPUT_DIR` pointed at the previous
   version's output dataset.
4. The analyzer / libclang are **not** used here: this notebook consumes an existing
   `base.json` and never regenerates it.
5. Every enrichment model is evaluated against the exact same base.json snapshot and
   the exact same frozen benchmark, so `summaries/comparison.csv` is a fair comparison.
6. **Multi-GPU:** on a session with two or more GPUs (e.g. Kaggle's T4 x2), a model
   that does not fit on one GPU is automatically sharded across all of them via
   accelerate (`device_map="auto"` + an explicit `max_memory`), not left unused --
   `check_fit()` decides this at runtime and GATE 1 prints the resulting
   `model.hf_device_map` so you can confirm both GPUs are in use for a 30B-class
   model. A model that fits on one GPU stays on one GPU even in a multi-GPU
   session. On a single-GPU session everything degrades to plain single-GPU loading.


## Cell 1 — Configuration

In [ ]:
import os
from pathlib import Path

# ---- Optima repository (this project) ----
OPTIMA_REPO_URL = "https://github.com/I1gorr/optima_python.git"
OPTIMA_REF = "main"  # pin to a commit SHA for full reproducibility

# ---- Filesystem roots ----
OPTIMA_DIR = Path("/kaggle/working/optima-python")
OUTPUT_ROOT = Path("/kaggle/working/optima_outputs")

# ---- base.json source: point this at your own analyzer output on GitHub.
# Prefer a commit-SHA raw URL (not a branch) for reproducibility, or pin
# BASE_JSON_EXPECTED_SHA256 below.
BASE_JSON_URL = "https://raw.githubusercontent.com/I1gorr/optima_python/main/output/base.json"
BASE_JSON_EXPECTED_SHA256 = None       # set to a sha256 hex digest to pin an exact snapshot
EXPECTED_FUNCTION_COUNT = 96           # set to None if your base.json legitimately differs

# ---- Benchmark (generated once from base.json only; never regenerated per-model) ----
BENCHMARK_JSON_URL = None              # None = generate from base.json; or a curated queries.json URL
NUM_QUERIES = 20
BENCHMARK_SEED = 42

# ---- Models: see optima_kaggle.models.MODEL_REGISTRY for the full, classified list ----
ACTIVE_MODEL = "qwen25-7b-instruct-nf4"
MODEL_QUEUE: list[str] = []             # additional slugs run after ACTIVE_MODEL (see RUN_QUEUE)
ALLOW_INFEASIBLE = False                # True unlocks Tier C registry entries on a single-GPU
                                         # session; on a 2+ GPU session check_fit() is the real gate
ALLOW_CPU_OFFLOAD_DEFAULT = False       # per-model ModelSpec.allow_cpu_offload default; a last
                                         # resort, orders of magnitude slower, flagged in outputs

# ---- Generation (centralized; identical across models for a fair comparison) ----
PROMPT_VARIANT = "colab_v2_no_module_ir"  # or "colab_v2" for the original Colab prompt (includes module-level LLVM IR)
MAX_INPUT_TOKENS_OVERRIDE = None          # None = use the model spec's registry default
MAX_NEW_TOKENS = 768
DO_SAMPLE = False                         # greedy decoding for deterministic, comparable evaluation
TEMPERATURE = None
TOP_P = None
RETRIES = 2

# ---- Full-run safety ----
MAX_CONSECUTIVE_FAILURES = 5
MIN_ENRICHMENT_SUCCESS_RATE = 0.95
MATERIALIZE_EVERY = 10
SMOKE_FUNCTION_IDS = None                 # e.g. ["path.cpp::func::12"] to pin GATE 3/4 functions
RUN_FULL_ENRICHMENT = True
RUN_QUEUE = False
STOP_ON_MODEL_FAILURE = True

# ---- Embedding / retrieval / evaluation (runs only after the GPU is free of any LLM) ----
RUN_EVALUATION = True
EMBEDDING_MODELS = ["bge-small"]
REPRESENTATION_MODES = ["hybrid", "semantic"]
K = 10

# ---- Resume (attach a previous version's /kaggle/working/optima_outputs as a dataset) ----
RESUME_INPUT_DIR = None   # e.g. Path("/kaggle/input/optima-outputs-v1/optima_outputs")

# ---- Environment / cleanup ----
INSTALL_BITSANDBYTES_IF_MISSING = True
DELETE_MODEL_CACHE_AFTER_UNLOAD = True

HANDLE = None  # the only variable ever allowed to hold a loaded model

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# CUDA_VISIBLE_DEVICES is deliberately left unset: both GPUs (e.g. a Kaggle
# T4 x2 session) must stay visible so a model that needs sharding can use
# them both. torch.cuda.device_count(), read in the diagnostics cell below,
# is what the rest of this notebook branches on -- never an env var.
os.environ.setdefault("HF_HOME", "/tmp/hf_home")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OPTIMA_EMBEDDING_DEVICE", "cuda")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "warning")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")


## Cell 2 — Kaggle environment diagnostics (pre-clone)

In [ ]:
import platform
import subprocess

print(f"Python version: {platform.python_version()}")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout)
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")


## Cell 3 — Clone / checkout Optima Python (sparse, shallow; no pip install)

In [ ]:
import subprocess
import sys


def _run(cmd, **kwargs):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result


if not (OPTIMA_DIR / ".git").is_dir():
    if OPTIMA_DIR.exists():
        raise RuntimeError(f"{OPTIMA_DIR} exists but is not a git repository; remove it first.")
    _run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
          OPTIMA_REPO_URL, str(OPTIMA_DIR)])
    _run(["git", "sparse-checkout", "set", "--cone", "optima", "colab", "optima_kaggle"], cwd=OPTIMA_DIR)
    _run(["git", "fetch", "--depth", "1", "origin", OPTIMA_REF], cwd=OPTIMA_DIR)
    _run(["git", "checkout", "FETCH_HEAD"], cwd=OPTIMA_DIR)
else:
    status = _run(["git", "status", "--porcelain"], cwd=OPTIMA_DIR)
    if status.stdout.strip():
        raise RuntimeError(f"{OPTIMA_DIR} has uncommitted changes; refusing to touch it.")
    _run(["git", "fetch", "--depth", "1", "origin", OPTIMA_REF], cwd=OPTIMA_DIR)
    _run(["git", "checkout", "FETCH_HEAD"], cwd=OPTIMA_DIR)

# A fresh kernel never has stale modules, but a re-run of this cell might.
for name in list(sys.modules):
    if name == "optima" or name.startswith(("optima.", "optima_kaggle", "colab")):
        del sys.modules[name]
if str(OPTIMA_DIR) not in sys.path:
    sys.path.insert(0, str(OPTIMA_DIR))

OPTIMA_COMMIT = _run(["git", "rev-parse", "HEAD"], cwd=OPTIMA_DIR).stdout.strip()
print(f"Optima checked out at {OPTIMA_DIR}, commit {OPTIMA_COMMIT}")


## Cell 4 — Install/check Python dependencies (never touches torch or clang)

In [ ]:
from optima_kaggle import environment as env

DEPS = env.ensure_python_deps(allow_install=True)


## Cell 5 — GPU/CUDA/internet diagnostics

In [ ]:
ENV_INFO = env.diagnose(hf_home=os.environ.get("HF_HOME"), working_dir="/kaggle/working")
NUM_GPUS = ENV_INFO["num_gpus"]
print(f"NUM_GPUS = {NUM_GPUS} (multi_gpu_capable={ENV_INFO['multi_gpu_capable']})")
env.gpu_report("session start")


## Cell 6 — bitsandbytes compatibility probe (fails loud, never silently downgrades)

In [ ]:
BNB = env.probe_bitsandbytes(INSTALL_BITSANDBYTES_IF_MISSING)
print(f"bitsandbytes: ok={BNB.ok}, stage={BNB.stage}, version={BNB.version}")
print(BNB.message)


## Cell 7 — Import Optima + optima_kaggle; verify the analyzer/libclang are not loaded

In [ ]:
import inspect
import sys as _sys

from colab import colab_pipeline
from optima.rag import embedding_simple
from optima_kaggle import enrichment, models, retrieval_eval, snapshot

assert "optima.analyzer" not in _sys.modules, "the analyzer must not be imported for this experiment"
assert "clang" not in _sys.modules and "clang.cindex" not in _sys.modules, "libclang must not be imported"

assert ".tmp" in inspect.getsource(colab_pipeline.save_json), (
    "colab_pipeline.save_json is not atomic on this checkout; push the local fix to "
    "colab/colab_pipeline.py on GitHub before running this notebook."
)
assert "OPTIMA_EMBEDDING_DEVICE" in inspect.getsource(embedding_simple), (
    "optima.rag.embedding_simple lacks OPTIMA_EMBEDDING_DEVICE support on this checkout; "
    "push the local fix on GitHub before running this notebook (otherwise embeddings run on CPU)."
)
print("Imports OK; analyzer/libclang not loaded; required upstream fixes are present.")


## Cell 8 — Download and snapshot base.json (downloaded exactly once per run)

In [ ]:
SNAP = snapshot.download_base_snapshot(BASE_JSON_URL, OUTPUT_ROOT, expected_sha256=BASE_JSON_EXPECTED_SHA256)

CONFIGURED_MODELS = [ACTIVE_MODEL, *MODEL_QUEUE] if RUN_QUEUE else [ACTIVE_MODEL]
CTX = snapshot.RunContext.create(
    OUTPUT_ROOT, SNAP,
    config={
        "prompt_variant": PROMPT_VARIANT, "max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE,
        "temperature": TEMPERATURE, "top_p": TOP_P, "retries": RETRIES,
        "embedding_models": EMBEDDING_MODELS, "representation_modes": REPRESENTATION_MODES, "k": K,
    },
    env=ENV_INFO, optima_commit=OPTIMA_COMMIT, configured_models=CONFIGURED_MODELS,
)
print(f"Run directory: {CTX.run_dir}")


## Cell 9 — Validate base.json

In [ ]:
BASE_REPORT = snapshot.validate_base_snapshot(SNAP.path, EXPECTED_FUNCTION_COUNT)


## Cell 10 — Restore resumable state from a previous run (no-op if RESUME_INPUT_DIR is None)

In [ ]:
snapshot.restore_resume_state(CTX, RESUME_INPUT_DIR)


## Cell 11 — Freeze the benchmark (generated once from base.json only; never from enriched JSON)

In [ ]:
BENCH = snapshot.freeze_benchmark(CTX, NUM_QUERIES, BENCHMARK_SEED, BENCHMARK_JSON_URL)


---
## Model stages (`ACTIVE_MODEL`)

GATE 1 (load) -> GATE 2 (trivial generation) -> GATE 3 (one real function) ->
GATE 4 (three real functions) -> full 96-function run. A failed gate raises and
stops here; it does not fall through to the full run.

## Cell 12 — Resolve the model spec, generation settings, and pre-load VRAM fit estimate

In [ ]:
from dataclasses import replace

SPEC = models.resolve_spec(ACTIVE_MODEL, num_gpus=NUM_GPUS, allow_infeasible=ALLOW_INFEASIBLE)
if MAX_INPUT_TOKENS_OVERRIDE is not None:
    SPEC = replace(SPEC, max_input_tokens=MAX_INPUT_TOKENS_OVERRIDE)
if SPEC.allow_cpu_offload is False and ALLOW_CPU_OFFLOAD_DEFAULT:
    SPEC = replace(SPEC, allow_cpu_offload=True)

GEN = enrichment.GenerationSettings(
    do_sample=DO_SAMPLE, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
    retries=RETRIES, prompt_variant=PROMPT_VARIANT,
)
# check_fit() is the sole feasibility gate (single_gpu_tier above was only an
# early, advisory refusal). It picks single-GPU vs sharded-across-both-GPUs
# placement and prints a per-GPU fit table.
FIT = models.check_fit(SPEC, num_gpus=NUM_GPUS)


## Cell 13 — GATE 1: model loads onto the GPU

In [ ]:
try:
    env.gpu_report("before load", CTX.gpu_log_path)
    HANDLE = models.load_model_safe(SPEC, BNB)
    env.gpu_report("after load", CTX.gpu_log_path)
    # load_model_safe() already printed the full hf_device_map; this is a
    # one-line summary so a sharded 30B-class load is obvious at a glance.
    print(f"placement={HANDLE.load_report['gpu_placement']}, "
          f"gpus_used={HANDLE.load_report['gpu_count_used']}, "
          f"used_cpu_offload={HANDLE.load_report['used_cpu_offload']}")
    G1 = enrichment.gate1_load(CTX, HANDLE, GEN)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 14 — GATE 2: trivial generation succeeds

In [ ]:
try:
    G2 = enrichment.gate2_trivial(CTX, HANDLE, GEN)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 15 — GATE 3: one real function from base.json is enriched

In [ ]:
try:
    G3 = enrichment.gate3_one_function(CTX, HANDLE, GEN, SNAP, SMOKE_FUNCTION_IDS)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 16 — GATE 4: three real functions are enriched (including the largest bounded prompt)

In [ ]:
try:
    G4 = enrichment.gate4_three_functions(CTX, HANDLE, GEN, SNAP, SMOKE_FUNCTION_IDS)
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 17 — Full resumable 96-function enrichment (only runs if GATES 1-4 passed)

In [ ]:
FULL = None
try:
    if RUN_FULL_ENRICHMENT:
        FULL = enrichment.run_full_enrichment(
            CTX, HANDLE, GEN, SNAP, MAX_CONSECUTIVE_FAILURES, MATERIALIZE_EVERY, MIN_ENRICHMENT_SUCCESS_RATE,
        )
        print(FULL["metrics"])
    else:
        print("RUN_FULL_ENRICHMENT is False; skipping the full run.")
except BaseException:
    models.emergency_unload(globals())
    raise


## Cell 18 — Unload the model and verify GPU memory is released

In [ ]:
models.unload_model(HANDLE, delete_cache=DELETE_MODEL_CACHE_AFTER_UNLOAD)
HANDLE = None
env.gpu_report("after unload", CTX.gpu_log_path)


## Cell 19 — Optional: run additional models sequentially (`MODEL_QUEUE`)

In [ ]:
QUEUE_RESULTS = None
if RUN_QUEUE and MODEL_QUEUE:
    def _gen_settings_factory(spec):
        return enrichment.GenerationSettings(
            do_sample=DO_SAMPLE, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS,
            retries=RETRIES, prompt_variant=PROMPT_VARIANT,
        )

    QUEUE_RESULTS = enrichment.run_model_queue(
        CTX, MODEL_QUEUE, _gen_settings_factory, BNB, num_gpus=NUM_GPUS,
        stop_on_failure=STOP_ON_MODEL_FAILURE,
        allow_infeasible=ALLOW_INFEASIBLE, delete_cache_after_unload=DELETE_MODEL_CACHE_AFTER_UNLOAD,
        max_consecutive_failures=MAX_CONSECUTIVE_FAILURES, materialize_every=MATERIALIZE_EVERY,
        min_success_rate=MIN_ENRICHMENT_SUCCESS_RATE,
    )
    print(QUEUE_RESULTS)
else:
    print("RUN_QUEUE is False or MODEL_QUEUE is empty; skipping the queue.")


---
## Embedding, retrieval, evaluation

Everything below runs with **no LLM on the GPU** and reuses the existing
`optima.rag` embedding/retrieval/evaluation implementation unchanged.

## Cell 20 — Discover which corpora passed enrichment; verify the GPU is free of any LLM

In [ ]:
assert HANDLE is None, "A model handle is still live; unload it before embedding/retrieval."
env.assert_gpu_clean(threshold_gib=0.3)
CORPORA = retrieval_eval.discover_passed_corpora(CTX)
print("Corpora to embed/evaluate:", CORPORA)


## Cell 21 — Build FAISS indexes for raw + every passed enrichment model

In [ ]:
INDEXES = None
if RUN_EVALUATION:
    INDEXES = retrieval_eval.build_indexes(CTX, CORPORA, EMBEDDING_MODELS, REPRESENTATION_MODES)


## Cell 22 — Retrieval smoke test (pipeline sanity, not a quality bar)

In [ ]:
if RUN_EVALUATION:
    from optima.rag.embedding_simple import embedding_alias
    retrieval_eval.retrieval_smoke(
        CTX, CORPORA, embedding_alias(EMBEDDING_MODELS[0]), REPRESENTATION_MODES[0], BENCH
    )


## Cell 23 — Full retrieval evaluation (Recall@K, MRR, latency) for every mode/alias/corpus

In [ ]:
MATRICES = None
if RUN_EVALUATION:
    MATRICES = retrieval_eval.evaluate_all(CTX, EMBEDDING_MODELS, REPRESENTATION_MODES, BENCH, K, CORPORA)


## Cell 24 — Cross-model comparison table (same base.json + benchmark for every row)

In [ ]:
COMPARISON = None
if RUN_EVALUATION:
    COMPARISON = retrieval_eval.build_comparison(CTX, MATRICES, CORPORA, BENCH)
    try:
        import pandas as pd
        display(pd.DataFrame(COMPARISON["rows"]).sort_values(
            ["representation_mode", "embedding_alias", "mrr"], ascending=[True, True, False]
        ))
    except ImportError:
        for row in COMPARISON["rows"]:
            print(row)


## Cell 25 — Final artifact report (raises if any configured model did not pass)

In [ ]:
snapshot.final_report(CTX)
